# A meal plan for tomorrow
## One original end-to-end Learn ML project

**Question:** At 16:00, can a fictional community-kitchen coordinator use information already available to estimate tomorrow's meals?

This is an optional continuation of the same six-lesson course. The explanations, code and synthetic data were independently authored for Code With Impact. No code, notebook cells, figures or data were copied from the referenced repositories. The notebook generates **180 fictional days**, independently of the small browser lesson datasets. Never compare their numerical scores.

**Execution status:** No saved outputs, so you see the results yourself. Tested on 26 October 2026 with Python 3.13, NumPy 2.5, pandas 3.0 and scikit-learn 1.9: it runs from start to finish.

**Workflow:** question → data contract → chronological split → inspection → training-only preparation → baseline → validation → freeze choice → final test → model note and monitoring.

## 1. Decide what would help

The coordinator uses a prediction to support a meal plan. Preparing too few meals and wasting surplus have different consequences, so a low average error is not the whole decision. A staff member still chooses a buffer and has authority to override the suggestion.

Before looking at scores, use this exercise rule: prefer a candidate only if it improves validation mean absolute error (MAE) by **at least 2 meals** over a training-median baseline. This arbitrary threshold makes "usefully better" explicit; a real kitchen would set it with staff. Also inspect underestimates, because MAE treats shortages and surplus symmetrically.

Write your own brief in the companion browser worksheet. State the person, action, timing, target, error cost and fallback. This notebook is a teaching demonstration, not evidence that an operational model is safe or useful.
Passing the exercise threshold is not evidence of a reliable gain. Judge variation across further time-aware validation windows and practical consequences before making a service decision.


In [ ]:
from pathlib import Path
import json
import platform

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 142
MIN_USEFUL_GAIN = 2.0
OUTPUT_DIR = Path("kitchen-project-output")
environment = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
    "joblib": joblib.__version__,
}
print(json.dumps(environment, indent=2))

## 2. Make an honest data contract

One row represents the day whose demand we want to predict. At 16:00 on the previous day, we assume these inputs exist:

| Field | Meaning | Available before the decision? |
| --- | --- | --- |
| `date` | Day being planned | Yes; used for ordering, not a model input here |
| `visitors_estimate` | Approximate visitor interest collected by the planning cutoff | Yes; occasionally missing |
| `rain_forecast_mm` | Forecast rainfall for the planned day | Yes; a forecast, not observed later rainfall |
| `day_type` | Weekday or weekend | Yes |
| `meals_needed` | Actual demand realised after the planned day | **No; target only** |

An actual same-day visitor count or observed end-of-day waste would leak information unavailable at planning time. Missing estimates must be handled using development data, not filled from the completed dataset.

Our generator deliberately creates simple relationships, noise and gradual change. It does not represent real people's needs, demographic groups, holidays, stock limits or reliable weather uncertainty.

In [ ]:
rng = np.random.default_rng(SEED)
n_days = 180
dates = pd.date_range("2025-01-06", periods=n_days, freq="D")
day_number = np.arange(n_days)
is_weekend = dates.dayofweek >= 5

# Latent interest is only a simulation ingredient, never an input to the model.
interest = (
    85 + 12 * np.sin(day_number / 12.0)
    + 0.10 * day_number + rng.normal(0, 8, n_days)
)
visitor_estimates = np.maximum(
    25, np.rint(interest + rng.normal(0, 6, n_days))
)
rain_forecasts = np.round(rng.gamma(1.7, 1.8, n_days), 1)
meals_needed = np.maximum(
    0,
    np.rint(
        15 + 0.80 * interest + 18 * is_weekend
        - 1.6 * rain_forecasts + rng.normal(0, 7, n_days)
    ),
).astype(int)

# Missing values appear in different periods and remain missing until a pipeline
# learns its imputation value from the training block.
missing_rows = rng.choice(n_days, size=14, replace=False)
visitor_estimates[missing_rows] = np.nan

days = pd.DataFrame({
    "date": dates,
    "visitors_estimate": visitor_estimates,
    "rain_forecast_mm": rain_forecasts,
    "day_type": np.where(is_weekend, "weekend", "weekday"),
    "meals_needed": meals_needed,
})
assert len(days) == 180
assert days["date"].is_monotonic_increasing

## 3. Protect later days before investigating

We want to plan the future, so keep time order rather than mixing later and earlier days. Use the first 60% to learn, the next 20% to compare candidates, and the last 20% for one final check.

This is **108 training / 36 validation / 36 final-test days**. Do not inspect final-test targets to pick features, preprocessing settings or model complexity. The data are synthetic and visible in code, so this is a discipline exercise rather than a security barrier.

A larger project should compare repeated forward validation windows inside its development period. Random folds would answer a different question when demand changes over time.

In [ ]:
train_end = int(0.60 * len(days))
validation_end = int(0.80 * len(days))
train = days.iloc[:train_end].copy()
validation = days.iloc[train_end:validation_end].copy()
final_test = days.iloc[validation_end:].copy()

assert (len(train), len(validation), len(final_test)) == (108, 36, 36)
assert train["date"].max() < validation["date"].min()
assert validation["date"].max() < final_test["date"].min()

split_summary = pd.DataFrame([
    {"block": name, "rows": len(block),
     "first_day": block["date"].min().date(),
     "last_day": block["date"].max().date()}
    for name, block in [
        ("training", train), ("validation", validation), ("final test", final_test)
    ]
])
print(split_summary.to_string(index=False))

## 4. Inspect training data and name the limits

Look at distributions, missingness and weekday/weekend coverage. Compare mean and median: are a few unusual days driving the mean? Could unrecorded demand be confused with low demand?

The table cannot establish that a model will work for every community or season. Even perfect weekend coverage in these fictional days does not guarantee representative data for a real kitchen.

In [ ]:
print("First training rows:")
print(train.head(6).to_string(index=False))
print("\nTraining numeric summary:")
print(train[["visitors_estimate", "rain_forecast_mm", "meals_needed"]]
      .describe().round(2))
print("\nMissing values in training inputs:")
print(train[["visitors_estimate", "rain_forecast_mm", "day_type"]].isna().sum())
print("\nTraining coverage and demand:")
print(train.groupby("day_type")["meals_needed"].agg(["size", "mean", "median"])
      .round(2))

## 5. Prepare inside the training pipeline

Use median imputation for numeric inputs, standardise them, and turn the weekday/weekend category into indicator columns. A `ColumnTransformer` applies the right preparation to each kind of column. Each model gets its own cloned preprocessing object.

Calling `fit` on a pipeline learns both preparation and the model **from training rows only**. Calling `predict` on later blocks reuses those learned values. It does not recompute their means, medians or category mapping. Unknown future categories are ignored by the encoder; that prevents a crash, but should also trigger a data-quality review.

Scaling is useful for Ridge. The forest does not require scaling, but the shared preparation keeps this small comparison consistent. A training-median `DummyRegressor` is the baseline because the median minimises absolute error for a constant prediction.

In [ ]:
numeric_features = ["visitors_estimate", "rain_forecast_mm"]
categorical_features = ["day_type"]
features = numeric_features + categorical_features
target = "meals_needed"

numeric_preparation = Pipeline([
    ("fill_missing", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
categorical_preparation = Pipeline([
    ("fill_missing", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preparation = ColumnTransformer([
    ("numeric", numeric_preparation, numeric_features),
    ("categorical", categorical_preparation, categorical_features),
])

estimators = {
    "median_baseline": DummyRegressor(strategy="median"),
    "ridge": Ridge(alpha=8.0),
    "small_forest": RandomForestRegressor(
        n_estimators=120, max_depth=4, min_samples_leaf=6,
        random_state=SEED, n_jobs=1,
    ),
}
models = {
    name: Pipeline([("prepare", clone(preparation)), ("predict", estimator)])
    for name, estimator in estimators.items()
}

X_train, y_train = train[features], train[target]
X_validation, y_validation = validation[features], validation[target]
for model in models.values():
    model.fit(X_train, y_train)

# A narrow check against accidental preparation on all rows.
learned_medians = models["ridge"].named_steps["prepare"].named_transformers_[
    "numeric"
].named_steps["fill_missing"].statistics_
assert np.allclose(
    learned_medians, X_train[numeric_features].median().to_numpy()
)
print("All three pipelines fitted on", len(train), "training rows only.")

## 6. Compare on identical validation days

MAE is the average absolute error, measured in meals here. Mean signed error (`prediction - actual`) adds direction; negative values mean underestimation on average. Also inspect the fraction of days underestimated. Neither number tells you how severe each shortage is, so look at individual errors.

Choose using validation, never the final-test score. These three settings are fixed for the exercise; repeated manual tuning against this small validation block would gradually overfit it. A complex model is not guaranteed to beat a simple one.

In [ ]:
validation_predictions = {
    name: model.predict(X_validation) for name, model in models.items()
}
validation_scores = pd.DataFrame([
    {
        "model": name,
        "mae_meals": mean_absolute_error(y_validation, prediction),
        "mean_signed_error": float(np.mean(prediction - y_validation.to_numpy())),
        "underestimated_days_fraction": float(
            np.mean(prediction < y_validation.to_numpy())
        ),
    }
    for name, prediction in validation_predictions.items()
]).set_index("model")
print(validation_scores.round(3).to_string())

baseline_mae = float(validation_scores.loc["median_baseline", "mae_meals"])
candidate_scores = validation_scores.drop(index="median_baseline")["mae_meals"]
best_candidate = candidate_scores.idxmin()
validation_gain = baseline_mae - float(candidate_scores.loc[best_candidate])
selected_name = (
    best_candidate if validation_gain >= MIN_USEFUL_GAIN else "median_baseline"
)
selection_reason = (
    f"{best_candidate} improves validation MAE by {validation_gain:.2f} meals; "
    f"required improvement is {MIN_USEFUL_GAIN:.1f}. "
    f"Selected {selected_name}."
)
print("\n" + selection_reason)

selected_model = models[selected_name]
validation_errors = pd.DataFrame({
    "date": validation["date"].to_numpy(),
    "actual_meals": y_validation.to_numpy(),
    "predicted_meals": validation_predictions[selected_name],
})
validation_errors["signed_error"] = (
    validation_errors["predicted_meals"] - validation_errors["actual_meals"]
)
print("\nLargest absolute validation errors:")
print(
    validation_errors.assign(
        absolute_error=validation_errors["signed_error"].abs()
    ).sort_values("absolute_error", ascending=False).head(6).round(2)
    .to_string(index=False)
)

## 7. Pause at the final-test gate

Before continuing, write down:

- Which model did the predeclared rule select? Did that match your guess?
- Is an average gain large enough to justify maintaining the model?
- Are the worst underestimates acceptable? Who should decide the planning buffer?
- Would another season, a school holiday or an input failure change the conclusion?
- Which data source did your worksheet result use?

**Freeze the choice now.** The next cell evaluates only the selected, already-fitted pipeline on the final 36 days. We deliberately do not refit on validation here, so the evaluated artifact is the same model we chose. Once the score is seen, the final block cannot fairly select a new design.

The in-memory guard prevents an accidental immediate repeat of the final cell. Restarting the kernel can reset it; it is not a secure lock. If you redesign after seeing final results, reserve genuinely new future data for a new final check.

In [ ]:
# Initialise the gate only once per kernel; rerunning this cell keeps the record.
if "final_result" not in globals():
    final_result = None

if final_result is not None:
    raise RuntimeError(
        "This kernel already opened the final test. Read the recorded result; "
        "do not use this block to select a different model."
    )

final_predictions = selected_model.predict(final_test[features])
final_actual = final_test[target].to_numpy()
final_result = {
    "selected_model": selected_name,
    "rows": len(final_test),
    "mae_meals": float(mean_absolute_error(final_actual, final_predictions)),
    "mean_signed_error": float(np.mean(final_predictions - final_actual)),
    "underestimated_days_fraction": float(np.mean(final_predictions < final_actual)),
}
print(json.dumps(final_result, indent=2))

## 8. Save what you actually evaluated

The saved artifact includes preprocessing and the chosen model. It should be used with the same named input columns and equivalent input meanings. The file is an exercise output, not a deployed service.

A final-test score is a limited estimate from one period. It can be worse than validation. Do not promise a particular MAE, a fixed ranking or reliable uncertainty intervals. Training on a synthetic formula cannot establish performance on real demand.

Only load model files you created or otherwise trust. Persisted Python model files can execute code when loaded and should be kept with their package-version record.

In [ ]:
if final_result is None:
    raise RuntimeError("Complete the final-test gate before saving artifacts.")
if final_result["selected_model"] != selected_name:
    raise RuntimeError("Model selection changed after the final test. Start a new study.")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(selected_model, OUTPUT_DIR / "kitchen-pipeline.joblib")
days.to_csv(OUTPUT_DIR / "synthetic-kitchen-days.csv", index=False)
(OUTPUT_DIR / "environment.json").write_text(
    json.dumps(environment, indent=2), encoding="utf-8"
)

# Only reload the model just created in this controlled local exercise.
reloaded_model = joblib.load(OUTPUT_DIR / "kitchen-pipeline.joblib")
assert np.allclose(
    reloaded_model.predict(X_validation.head(3)),
    selected_model.predict(X_validation.head(3)),
)
print("Saved selected pipeline, original synthetic data and environment record.")

## 9. Write the model note and a monitoring action

Name who acts when things change. In a real pilot, a coordinator might check weekly MAE, missing-input rate and the share of days with shortages against agreed limits. If errors rise for two weeks, investigate data changes with staff and use a manual planning rule until reviewed. Those limits would need to be agreed with the service.

Monitoring is useful only when there is a concrete response. The suggested plan below is a draft for discussion, not a validated operational policy.

In [ ]:
model_note = f"""# Community-kitchen learning model

## Decision and data
At 16:00, support the coordinator's meal plan for tomorrow.
All 180 daily rows are independently generated synthetic data.
Inputs: prior-day visitor estimate, rainfall forecast and known day type.
Actual demand is the target and is never an input.
Split in chronological order: 108 training, 36 validation, 36 final-test days.
Preprocessing and every candidate were fitted on training rows only.

## Selection and result
{selection_reason}
Validation MAE for the selected model:
{validation_scores.loc[selected_name, 'mae_meals']:.2f} meals.
One final-test MAE: {final_result['mae_meals']:.2f} meals.
Final mean signed error: {final_result['mean_signed_error']:.2f} meals.
Fraction of final days underestimated:
{final_result['underestimated_days_fraction']:.1%}.
The selected model was not refitted after validation.

## Limits
Synthetic formula, short single period and one final split.
No real service, demographic, holiday or supplier data.
MAE weights shortages and surplus equally; error costs need staff input.
No prediction interval or guarantee for tomorrow.
A prediction requires a human decision about buffers and capacity.

## Monitoring and response draft
Owner: the kitchen coordinator in a supervised pilot.
Check weekly MAE, shortage frequency and missing visitor estimates.
Agree numeric action thresholds with staff before a real pilot.
If errors exceed agreed limits for two weeks, inspect the data and
return to a staff-led manual plan until reviewed.
Review input availability before any deployment; collect new future
data for evaluation if modelling changes after this final test.

## Credits
Original explanation, code and generated data: Code With Impact.
Conceptual workflow reference: Aurélien Géron, Hands-On Machine Learning.
Sampling and evaluation references: Peter Bruce, Andrew Bruce and
Peter Gedeck, Practical Statistics for Data Scientists.
No code, notebook cells, figures or datasets copied from those repositories.
"""
(OUTPUT_DIR / "model-note.md").write_text(model_note, encoding="utf-8")
print(model_note)
print("Artifacts written to:", OUTPUT_DIR.resolve())

## References and next questions

- **Aurélien Géron:** [Hands-On Machine Learning, chapter 2 end-to-end project](https://github.com/ageron/handson-mlp/blob/main/02_end_to_end_machine_learning_project.ipynb). Conceptual workflow reference.
- **Peter Bruce, Andrew Bruce and Peter Gedeck:** [Practical Statistics for Data Scientists repository](https://github.com/gedeck/practical-statistics-for-data-scientists). Conceptual sampling and evaluation references.
- Official scikit-learn documentation: [pipelines and column transformers](https://scikit-learn.org/stable/modules/compose.html), [avoiding leakage](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage), [model persistence](https://scikit-learn.org/stable/model_persistence.html).
- Course [Sources & credits](../sources.html) and [reuse register](../CREDITS.md).

**Originality:** References acknowledge the ideas and authors that informed the teaching. This notebook's narrative, code and data generator were independently authored. No external repository code, notebook cells, figures or datasets were imported.

For further practice, design repeated forward validation windows within the first 144 development days, compare a weekend-specific baseline, or propose a shortage-weighted objective. Set the design before looking at a fresh final period. If the existing final block has already been opened, it is no longer a fair source of model-selection evidence.

Return to the browser worksheet and complete the model note in your own words. State which uncertainty needs new evidence rather than another algorithm.